In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import pandas as pd
from src.config import (
    load_config,
    get_embedding_model,
    get_reranker_model,
    get_top_k,
)

from src.knowledge_base import KnowledgeBaseService
from src.embeddings import EmbeddingService
from src.faiss_index import FaissIndexService
from src.retriever import RetrieverService
from src.reranker import RerankerService
from src.rag_pipeline import RAGPipeline

In [3]:
DATA_DIR = Path("../data/raw")
train = pd.read_csv(DATA_DIR / "train.csv")
config = load_config()

knowledge_base_service = KnowledgeBaseService()
knowledge_base_service.build(train)

embedding_service = EmbeddingService(
    get_embedding_model(config)
)

faiss_service = FaissIndexService()

faiss_service.load(
    PROJECT_ROOT /
    "outputs" /
    "faiss" /
    "knowledge_base.faiss"
)

retriever_service = RetrieverService(
    embedding_service,
    faiss_service,
)

reranker_service = RerankerService(
    get_reranker_model(config)
)

2026-07-17 11:54:04,966 | INFO | src.knowledge_base | Building Knowledge Base...
2026-07-17 11:54:04,980 | INFO | src.knowledge_base | Knowledge Base built successfully with 2000 documents.
2026-07-17 11:54:04,981 | INFO | src.embeddings | Loading embedding model: all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

2026-07-17 11:54:11,564 | INFO | src.embeddings | Embedding model loaded.
2026-07-17 11:54:11,566 | INFO | src.faiss_index | FAISS index loaded successfully.
2026-07-17 11:54:11,567 | INFO | src.reranker | Loading Cross Encoder model: cross-encoder/ms-marco-MiniLM-L-6-v2


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

2026-07-17 11:54:20,990 | INFO | src.reranker | Cross Encoder loaded successfully.


In [4]:
rag_pipeline = RAGPipeline(
    knowledge_base_service,
    retriever_service,
    reranker_service,
)

In [5]:
query = "What is machine learning?"

results = rag_pipeline.retrieve_context(
    query,
    get_top_k(config),
)

2026-07-17 11:54:21,120 | INFO | src.rag_pipeline | Executing RAG pipeline.
2026-07-17 11:54:21,120 | INFO | src.retriever | Embedding query.
2026-07-17 11:54:21,121 | INFO | src.embeddings | Generating embeddings for 1 documents.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-07-17 11:54:21,221 | INFO | src.embeddings | Embedding generation completed.
2026-07-17 11:54:21,222 | INFO | src.retriever | Searching FAISS index.
2026-07-17 11:54:21,224 | INFO | src.retriever | Retrieved 10 documents.
2026-07-17 11:54:21,224 | INFO | src.reranker | Reranking 10 retrieved documents.
2026-07-17 11:54:21,269 | INFO | src.reranker | Reranking completed.
2026-07-17 11:54:21,269 | INFO | src.rag_pipeline | RAG pipeline completed.


In [6]:
for rank, result in enumerate(results, start=1):

    print(f"Rank {rank}")
    print(f"Score : {result['score']:.4f}")
    print(result["document"][:200])
    print("-" * 80)

Rank 1
Score : -6.0902
A device used to demonstrate a neuro-inspired circuit that shows long-term potentiation for learning and inactivity-based forgetting.
--------------------------------------------------------------------------------
Rank 2
Score : -6.0902
A device used to demonstrate a neuro-inspired circuit that shows long-term potentiation for learning and inactivity-based forgetting.
--------------------------------------------------------------------------------
Rank 3
Score : -6.0902
A device used to demonstrate a neuro-inspired circuit that shows long-term potentiation for learning and inactivity-based forgetting.
--------------------------------------------------------------------------------
Rank 4
Score : -6.0902
A device used to demonstrate a neuro-inspired circuit that shows long-term potentiation for learning and inactivity-based forgetting.
--------------------------------------------------------------------------------
Rank 5
Score : -6.0902
A device used to demonstr

In [7]:
assert len(results) == get_top_k(config)

assert results[0]["score"] >= results[-1]["score"]

In [8]:
test = pd.read_csv(DATA_DIR / "test.csv")
row = test.iloc[0]

prediction = rag_pipeline.predict(
    row,
    top_k=3,
)

print(prediction)

2026-07-17 11:54:21,313 | INFO | src.rag_pipeline | Predicting answer options.
2026-07-17 11:54:21,313 | INFO | src.rag_pipeline | Executing RAG pipeline.
2026-07-17 11:54:21,314 | INFO | src.retriever | Embedding query.
2026-07-17 11:54:21,314 | INFO | src.embeddings | Generating embeddings for 1 documents.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-07-17 11:54:21,330 | INFO | src.embeddings | Embedding generation completed.
2026-07-17 11:54:21,331 | INFO | src.retriever | Searching FAISS index.
2026-07-17 11:54:21,331 | INFO | src.retriever | Retrieved 3 documents.
2026-07-17 11:54:21,331 | INFO | src.reranker | Reranking 3 retrieved documents.
2026-07-17 11:54:21,350 | INFO | src.reranker | Reranking completed.
2026-07-17 11:54:21,351 | INFO | src.rag_pipeline | RAG pipeline completed.
2026-07-17 11:54:21,351 | INFO | src.rag_pipeline | Prediction completed.


['A', 'B', 'C']


In [9]:
predictions = []

for _, row in test.iterrows():

    prediction = rag_pipeline.predict(
        row,
        top_k=3,
    )

    predictions.append(
        " ".join(prediction)
    )

2026-07-17 11:54:21,384 | INFO | src.rag_pipeline | Predicting answer options.
2026-07-17 11:54:21,386 | INFO | src.rag_pipeline | Executing RAG pipeline.
2026-07-17 11:54:21,386 | INFO | src.retriever | Embedding query.
2026-07-17 11:54:21,386 | INFO | src.embeddings | Generating embeddings for 1 documents.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-07-17 11:54:21,408 | INFO | src.embeddings | Embedding generation completed.
2026-07-17 11:54:21,410 | INFO | src.retriever | Searching FAISS index.
2026-07-17 11:54:21,411 | INFO | src.retriever | Retrieved 3 documents.
2026-07-17 11:54:21,411 | INFO | src.reranker | Reranking 3 retrieved documents.
2026-07-17 11:54:21,426 | INFO | src.reranker | Reranking completed.
2026-07-17 11:54:21,426 | INFO | src.rag_pipeline | RAG pipeline completed.
2026-07-17 11:54:21,427 | INFO | src.rag_pipeline | Prediction completed.
2026-07-17 11:54:21,427 | INFO | src.rag_pipeline | Predicting answer options.
2026-07-17 11:54:21,427 | INFO | src.rag_pipeline | Executing RAG pipeline.
2026-07-17 11:54:21,427 | INFO | src.retriever | Embedding query.
2026-07-17 11:54:21,428 | INFO | src.embeddings | Generating embeddings for 1 documents.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-07-17 11:54:21,448 | INFO | src.embeddings | Embedding generation completed.
2026-07-17 11:54:21,448 | INFO | src.retriever | Searching FAISS index.
2026-07-17 11:54:21,449 | INFO | src.retriever | Retrieved 3 documents.
2026-07-17 11:54:21,449 | INFO | src.reranker | Reranking 3 retrieved documents.
2026-07-17 11:54:21,466 | INFO | src.reranker | Reranking completed.
2026-07-17 11:54:21,467 | INFO | src.rag_pipeline | RAG pipeline completed.
2026-07-17 11:54:21,467 | INFO | src.rag_pipeline | Prediction completed.
2026-07-17 11:54:21,467 | INFO | src.rag_pipeline | Predicting answer options.
2026-07-17 11:54:21,467 | INFO | src.rag_pipeline | Executing RAG pipeline.
2026-07-17 11:54:21,468 | INFO | src.retriever | Embedding query.
2026-07-17 11:54:21,468 | INFO | src.embeddings | Generating embeddings for 1 documents.


...

2026-07-17 11:54:35,690 | INFO | src.embeddings | Embedding generation completed.
2026-07-17 11:54:35,690 | INFO | src.retriever | Searching FAISS index.
2026-07-17 11:54:35,691 | INFO | src.retriever | Retrieved 3 documents.
2026-07-17 11:54:35,691 | INFO | src.reranker | Reranking 3 retrieved documents.
2026-07-17 11:54:35,704 | INFO | src.reranker | Reranking completed.
2026-07-17 11:54:35,704 | INFO | src.rag_pipeline | RAG pipeline completed.
2026-07-17 11:54:35,705 | INFO | src.rag_pipeline | Prediction completed.
2026-07-17 11:54:35,705 | INFO | src.rag_pipeline | Predicting answer options.
2026-07-17 11:54:35,705 | INFO | src.rag_pipeline | Executing RAG pipeline.
2026-07-17 11:54:35,705 | INFO | src.retriever | Embedding query.
2026-07-17 11:54:35,705 | INFO | src.embeddings | Generating embeddings for 1 documents.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-07-17 11:54:35,714 | INFO | src.embeddings | Embedding generation completed.
2026-07-17 11:54:35,714 | INFO | src.retriever | Searching FAISS index.
2026-07-17 11:54:35,714 | INFO | src.retriever | Retrieved 3 documents.
2026-07-17 11:54:35,715 | INFO | src.reranker | Reranking 3 retrieved documents.
2026-07-17 11:54:35,728 | INFO | src.reranker | Reranking completed.
2026-07-17 11:54:35,728 | INFO | src.rag_pipeline | RAG pipeline completed.
2026-07-17 11:54:35,729 | INFO | src.rag_pipeline | Prediction completed.
2026-07-17 11:54:35,729 | INFO | src.rag_pipeline | Predicting answer options.
2026-07-17 11:54:35,729 | INFO | src.rag_pipeline | Executing RAG pipeline.
2026-07-17 11:54:35,729 | INFO | src.retriever | Embedding query.
2026-07-17 11:54:35,729 | INFO | src.embeddings | Generating embeddings for 1 documents.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-07-17 11:54:35,740 | INFO | src.embeddings | Embedding generation completed.
2026-07-17 11:54:35,740 | INFO | src.retriever | Searching FAISS index.
2026-07-17 11:54:35,741 | INFO | src.retriever | Retrieved 3 documents.
2026-07-17 11:54:35,741 | INFO | src.reranker | Reranking 3 retrieved documents.
2026-07-17 11:54:35,752 | INFO | src.reranker | Reranking completed.
2026-07-17 11:54:35,752 | INFO | src.rag_pipeline | RAG pipeline completed.
2026-07-17 11:54:35,752 | INFO | src.rag_pipeline | Prediction completed.


In [10]:

submission = test[["id"]].copy()

submission["prediction"] = predictions

submission.head()

,id,prediction
0,1,A B C
1,2,A B C
2,3,B A C
3,4,E A B
4,5,C A B


In [11]:
submission.to_csv(
    PROJECT_ROOT /
    "outputs" /
    "submission.csv",
    index=False,
)